In [ ]:
from pyspark.sql import SparkSession
from dotenv import load_dotenv
from os import getenv
load_dotenv()
spark = SparkSession.builder \
    .appName(getenv("HOSTED_BOOK_ETL_SERVER_URL")) \
    .getOrCreate()

# Read scraped JSON or API output
books_df = spark.read.json("gs://scrape-bucket/books/*.json") # COULD BE .jl-postprocess, .csv, parquet or others !

# Read MySQL data (e.g., existing table)
mysql_df = spark.read.format("jdbc").options(
    url="jdbc:mysql://localhost:3306/LiteratureScrapeDB",
    dbtable="BookReview",
    user=getenv("BOOK_ETL_user"),
    password=getenv("BOOK_ETL_pwd")
).load()


In [ ]:
# Clean + merge - VERY BASIC EDA and cleaning Pandas-like
combined_df = books_df.join(mysql_df, "book_id", "left")
cleaned_df = combined_df.dropna(subset=["title", "avgRating"])

cleaned_df = cleaned_df.repartition(8, "publishedDate")

cleaned_df.write.format("bigquery") \
    .option("table", "project.dataset.BookReview") \
    .option("partitionField", "publishedDate") \
    .option("clusteredFields", "book_id, reviewer_id") \
    .mode("overwrite") \
    .save()

mysql_df = spark.read.format("jdbc").options(
    url="jdbc:mysql://localhost:3306/LiteratureScrapeDB",
    dbtable="BookReview",
    user="root",
    password="yourpass"
).load()

mysql_df.write.format("bigquery") \
    .option("table", "project.dataset.BookReview") \
    .mode("overwrite") \
    .save()

In [ ]:
# Read scraped Books
books_df = spark.read.json("gs://literature_scrape/raw/books/*.json")

# Merge & clean
merged_df = books_df.join(authors_df, books_df.author_gid == authors_df.author_gid, "left") \
    .withColumn("rating", when(col("rating").isNull(), 0).otherwise(col("rating"))) \
    .dropDuplicates(["book_id"])

# Write to BigQuery
merged_df.write.format("bigquery") \
    .option("table", "project.dataset.BookFull") \
    .option("partitionField", "publishedDate") \
    .option("clusteredFields", "language, averageRating") \
    .mode("overwrite") \
    .save()